<a href="https://colab.research.google.com/github/JGradone/Topics-Sentinel/blob/main/Activity_3_Redwing_Science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Activity 3 - Redwing Science Data
**Rutgers University Topics in Marine Science**

*Joe Gradone, Rutgers University*

**Redwing** is a Teledyne Webb Research Slocum glider flying the Sentinel mission, a Rutgers / Teledyne Webb attempt to be the first autonomous underwater vehicle to circumnavigate the globe. On every dive and climb its **CTD** (Conductivity–Temperature–Depth sensor) measured the ocean. In this notebook you'll use those measurements to see:

* how temperature, salinity and density change with depth and along the track,
* which **water masses** Redwing flew through,
* and how the sensor itself can bend the data (**thermal lag**).

How the glider *flies* is in **Activity 2**. This notebook uses the same `load_glider_data()` function from Activities 1 and 2.

### Learning goals
By the end you should be able to:
* Make depth–time sections, profiles, and T–S diagrams
* Identify water masses on a T–S diagram
* Compare a dive with the climb after it and explain the differences

### How to use this notebook
**Run the cells in order from the top** (Shift + Enter). Later cells use variables made in earlier ones. If something breaks, use *Runtime → Restart session and run all*.

Cells marked **✏️ Your turn** are for you. Change the values marked `# <- change me` and see what happens.

---
## 1. Setup

Colab already has `numpy`, `pandas`, and `matplotlib`. We install three more:
* [`erddapy`](https://ioos.github.io/erddapy/): builds ERDDAP data requests for us
* [`gsw`](https://teos-10.github.io/GSW-Python/): the TEOS-10 seawater equation of state (density and more)
* [`cmocean`](https://matplotlib.org/cmocean/): colormaps designed for oceanography

In [ ]:
%pip install --quiet erddapy gsw cmocean

In [ ]:
import io
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from erddapy import ERDDAP
import matplotlib.dates as mdates
import cmocean.cm as cmo
import gsw

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

---
## 2. The data: Redwing's delayed-mode science profiles

We use the **delayed-mode** science dataset, `SCIENCE_ID` (`redwing-20251011T1511-profile-sci-delayed`). [Browse it here.](https://slocum-data.marine.rutgers.edu/erddap/tabledap/redwing-20251011T1511-profile-sci-delayed.html) It's the complete CTD record from the glider (one measurement per second), already processed:

* every dive and every climb has its own **`profile_id`**,
* **salinity** and **density** are already calculated.

The next cell defines the same functions you used in Activities 1 and 2.

In [ ]:
SERVER = 'https://slocum-data.marine.rutgers.edu/erddap'
ENGINEERING_ID = 'redwing-20251011T1511-trajectory-raw-delayed'  # everything the glider logged, ~once a second
SCIENCE_ID = 'redwing-20251011T1511-profile-sci-delayed'         # processed CTD profiles: salinity, density, ...


def read_csv_from_url(url, **kwargs):
    """Download a CSV file from the web into a DataFrame.

    Uses `requests`, which safely encodes characters like < and > in the web address.
    If Python can't verify the server's SSL certificate, it retries without checking it.
    That's fine for public, read-only data like this."""
    try:
        response = requests.get(url, timeout=600)
    except requests.exceptions.SSLError:
        requests.packages.urllib3.disable_warnings()
        response = requests.get(url, verify=False, timeout=600)
    if not response.ok:  # show ERDDAP's explanation, e.g. "Your query produced no matching results"
        raise RuntimeError(f'ERDDAP error {response.status_code}: {response.text[:500]}')
    return pd.read_csv(io.StringIO(response.text), **kwargs)


def load_glider_data(dataset_id, variables, start=None, end=None, filters=None, server=SERVER):
    """Download glider variables from ERDDAP and return a pandas DataFrame.

    dataset_id : which dataset, e.g. ENGINEERING_ID or SCIENCE_ID
    variables  : list of variable names, e.g. ['time', 'm_depth']
    start, end : optional time limits (UTC), e.g. '2025-10-20' or '2025-10-20 06:00'
    filters    : optional list of extra ERDDAP filters that run on the server, e.g.
                 ['m_depth!=NaN'] keeps only rows where m_depth has a value
    """
    if 'time' not in variables:
        variables = ['time'] + list(variables)

    e = ERDDAP(server=server, protocol='tabledap', response='csv')
    e.dataset_id = dataset_id
    e.variables = variables
    constraints = {}
    if start is not None:
        constraints['time>='] = pd.Timestamp(start).strftime('%Y-%m-%dT%H:%M:%SZ')
    if end is not None:
        constraints['time<='] = pd.Timestamp(end).strftime('%Y-%m-%dT%H:%M:%SZ')
    e.constraints = constraints

    url = e.get_download_url()
    for f in filters or []:
        url += '&' + quote(f, safe='=!,()/')
    df = read_csv_from_url(url, skiprows=[1])  # ERDDAP puts units in the 2nd row; skip it
    df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)  # all times are UTC
    return df.sort_values('time').reset_index(drop=True)


def list_glider_variables(dataset_id, server=SERVER):
    """Return a table of every variable in a dataset with its units and long name."""
    e = ERDDAP(server=server, protocol='tabledap')
    info = read_csv_from_url(e.get_info_url(dataset_id, response='csv'))
    attrs = info[info['Row Type'] == 'attribute']
    table = attrs.pivot_table(index='Variable Name', columns='Attribute Name',
                              values='Value', aggfunc='first')
    names = info.loc[info['Row Type'] == 'variable', 'Variable Name']
    return table.reindex(names)[['units', 'long_name']]

### Variables we'll use

| Variable | What it is | Units |
|---|---|---|
| `profile_id` | which dive or climb this measurement belongs to | (a number) |
| `depth` | depth of the CTD | m |
| `pressure` | water pressure | dbar (1 dbar ≈ 1 m) |
| `temperature` | water temperature | °C |
| `salinity` | practical salinity (calculated from conductivity, temperature and pressure) | no units |
| `density` | seawater density | kg/m³ |
| `sound_speed` | speed of sound in seawater | m/s |

Run the next cell to see every variable in the dataset.

In [ ]:
list_glider_variables(SCIENCE_ID)

---
## 3. One day at full resolution

Let's start small: one day of every measurement.

In [ ]:
# ✏️ Pick a day (Redwing dove from Oct 11, 2025 to Jan 16, 2026)
start = '2025-11-15'  # <- change me
end   = '2025-11-16'  # <- change me

day = load_glider_data(SCIENCE_ID,
                       ['profile_id', 'depth', 'temperature', 'salinity', 'density'],
                       start=start, end=end)
print(f'{len(day):,} measurements in {day.profile_id.nunique()} profiles')
day.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sc = ax.scatter(day.time, day.depth, c=day.temperature, s=1, cmap=cmo.thermal)
fig.colorbar(sc, ax=ax, label='Temperature (°C)')
ax.invert_yaxis()
ax.set_ylabel('Depth (m)')
ax.set_title(f'Redwing temperature, {start} to {end}')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d\n%H:%M'))
plt.show()

---
## 4. The whole mission

The full science record is millions of measurements, too many to download comfortably. Instead we ask ERDDAP to **bin-average** each profile into 5 dbar layers **on the server**, with the filter `'orderByMean("profile_id,pressure/5")'`.

That shrinks each profile from ~3,000 measurements to ~180 averages. It takes about 30 seconds.

In [ ]:
binned = load_glider_data(SCIENCE_ID,
                          ['profile_id', 'latitude', 'longitude', 'pressure', 'depth',
                           'temperature', 'salinity', 'density', 'sound_speed'],
                          filters=['orderByMean("profile_id,pressure/5")'])

# After binning, ERDDAP puts the bin *number* in the pressure column (0 = 0-5 dbar, 1 = 5-10 dbar, ...),
# so we drop it and use the bin-averaged depth instead
binned = binned.drop(columns='pressure')

# Potential density anomaly (sigma-0): density - 1000 kg/m3, as if the water were brought to the surface.
# It removes the squeezing effect of pressure so we can compare water from different depths.
p = gsw.p_from_z(-binned.depth, binned.latitude)
SA = gsw.SA_from_SP(binned.salinity, p, binned.longitude, binned.latitude)
binned['cons_temp'] = gsw.CT_from_t(SA, binned.temperature, p)
binned['sigma0'] = gsw.sigma0(SA, binned.cons_temp)

print(f'{len(binned):,} bin averages from {binned.profile_id.nunique():,} profiles')
binned.head()

### A table of profiles

Let's summarize each profile: when and where it was, how deep it went, and whether it was a **dive** (deepest point comes *after* the shallowest) or a **climb**.

In [ ]:
g = binned.groupby('profile_id')
profiles = pd.DataFrame({
    'start': g.time.min(),
    'end': g.time.max(),
    'latitude': g.latitude.mean(),
    'longitude': g.longitude.mean(),
    'max_depth': g.depth.max(),
})
time_at_deepest = binned.loc[g.depth.idxmax()].set_index('profile_id').time
time_at_shallowest = binned.loc[g.depth.idxmin()].set_index('profile_id').time
profiles['direction'] = np.where(time_at_deepest > time_at_shallowest, 'dive', 'climb')

print(f'{len(profiles):,} profiles')
print(f"{(profiles.max_depth > 800).sum()} go deeper than 800 m")
profiles.head()

🤔 Most profiles only go to about 25 m. Plot `profiles.max_depth` against `profiles.start` to see when the deep and shallow ones happen. Why might a glider make lots of shallow yos near the surface?

### Depth–time sections

A **section** shows the whole water column changing through time (and, since the glider is moving, along the track). Color = the measured property.

In [ ]:
# ✏️ Pick a time window for the section
start = '2025-10-11'  # <- change me
end   = '2026-01-17'  # <- change me

s = binned[(binned.time >= start) & (binned.time <= end)]

fig, axs = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
for ax, var, cmap, label in zip(
        axs,
        ['temperature', 'salinity', 'sigma0'],
        [cmo.thermal, cmo.haline, cmo.dense],
        ['Temperature (°C)', 'Salinity', 'σ₀ (kg/m³)']):
    vmin, vmax = s[var].quantile([0.01, 0.99])
    sc = ax.scatter(s.time, s.depth, c=s[var], s=2, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(sc, ax=ax, label=label)
    ax.invert_yaxis()
    ax.set_ylabel('Depth (m)')
fig.suptitle(f'Redwing CTD sections, {start} to {end}')
fig.tight_layout()
plt.show()

### A map at one depth

Sections mix up changes in time and changes in place. A map of one depth along the track shows *where* the water changed.

In [ ]:
# ✏️ Pick a depth (m)
target_depth = 200  # <- change me

layer = binned[(binned.depth - target_depth).abs() < 2.5]

fig, ax = plt.subplots(figsize=(12, 5))
sc = ax.scatter(layer.longitude, layer.latitude, c=layer.temperature, s=8, cmap=cmo.thermal)
fig.colorbar(sc, ax=ax, label='Temperature (°C)')
ax.set_aspect(1 / np.cos(np.radians(layer.latitude.mean())))
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title(f'Temperature at {target_depth} m')
plt.show()

🤔 **Discuss:** Where does the temperature at 200 m jump? That sharp boundary is a **front**. Which current is it? (Compare with the current map in Activity 2.)

---
## 5. T–S diagram: finding water masses

Plotting temperature against salinity lets us identify **water masses**: bodies of water with a characteristic temperature and salinity, set where they formed at the surface. Water keeps its T–S "fingerprint" for a long time after it sinks. The grey lines are constant density (σ₀).

In [ ]:
s = binned[(binned.time >= start) & (binned.time <= end)]

# density contours for the background
ct_grid, sp_grid = np.meshgrid(np.linspace(s.cons_temp.min() - 1, s.cons_temp.max() + 1, 100),
                               np.linspace(s.salinity.min() - 0.2, s.salinity.max() + 0.2, 100))
sig_grid = gsw.sigma0(sp_grid, ct_grid)  # using salinity as an approximation of SA for the contours

fig, ax = plt.subplots(figsize=(8, 7))
cs = ax.contour(sp_grid, ct_grid, sig_grid, colors='0.6', linewidths=0.8, levels=np.arange(20, 30, 0.5))
ax.clabel(cs, fmt='%.1f', fontsize=8)
sc = ax.scatter(s.salinity, s.cons_temp, c=s.depth, s=2, cmap=cmo.deep)
fig.colorbar(sc, ax=ax, label='Depth (m)')
ax.set_xlabel('Salinity')
ax.set_ylabel('Conservative temperature (°C)')
ax.set_title('T–S diagram')
plt.show()

🤔 **Discuss:**
* The deep water (dark points) falls on one tight line. Why is deep water so much more uniform than surface water?
* Look for a cluster near **18 °C and 36.5**. That's **Eighteen Degree Water**, a famous water mass formed each winter south of the Gulf Stream. At what depths did Redwing find it?

---
## 6. A dive and the climb right after it

Now we zoom in on one dive and the climb that follows, at **full resolution**. The glider is in nearly the same place, so both should show the same ocean. Differences come from real ocean changes, or from the sensor: the conductivity cell takes time to warm up or cool down to the water around it (**thermal lag**). That makes the calculated salinity wrong where temperature changes quickly, like in the thermocline.

First, list the full-depth dives that have a climb right after them.

In [ ]:
next_is_climb = ((profiles.direction.shift(-1) == 'climb')
                 & (profiles.start.shift(-1) - profiles.end < pd.Timedelta('10min')))
deep_dives = profiles[(profiles.direction == 'dive') & (profiles.max_depth > 800) & next_is_climb]
print(f'{len(deep_dives)} full-depth dive + climb pairs. The first few:')
deep_dives.head()

In [ ]:
def load_dive_climb(dive_id):
    """Download a dive and the climb after it at full resolution."""
    climb_id = profiles.index[profiles.index.get_loc(dive_id) + 1]
    t0 = profiles.loc[dive_id, 'start'] - pd.Timedelta('1min')
    t1 = profiles.loc[climb_id, 'end'] + pd.Timedelta('1min')
    df = load_glider_data(SCIENCE_ID, ['profile_id', 'depth', 'temperature', 'salinity', 'density'],
                          start=t0, end=t1)
    dive = df[df.profile_id == dive_id]
    climb = df[df.profile_id == climb_id]
    return dive, climb


def plot_dive_climb(dive, climb):
    """Plot temperature, salinity and density for a dive and a climb."""
    fig, axs = plt.subplots(1, 3, figsize=(13, 7), sharey=True)
    for ax, var, label in zip(axs, ['temperature', 'salinity', 'density'],
                              ['Temperature (°C)', 'Salinity', 'Density (kg/m³)']):
        ax.plot(dive[var], dive.depth, '-', lw=1, label='dive')
        ax.plot(climb[var], climb.depth, '-', lw=1, label='climb')
        ax.set_xlabel(label)
    axs[0].invert_yaxis()
    axs[0].set_ylabel('Depth (m)')
    axs[0].legend(loc='lower right')
    fig.suptitle(f'Dive and climb starting {dive.time.min():%Y-%m-%d %H:%M} UTC')
    fig.tight_layout()
    plt.show()


# ✏️ Pick a pair: deep_dives.index[0] is the first full-depth dive, [1] the second, ...
dive, climb = load_dive_climb(deep_dives.index[20])  # <- change me
print(f'{len(dive):,} dive and {len(climb):,} climb measurements')
plot_dive_climb(dive, climb)

Let's zoom in on the top 300 m, where most of the action is.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
for ax, var, label in zip(axs, ['temperature', 'salinity'], ['Temperature (°C)', 'Salinity']):
    ax.plot(dive[var], dive.depth, '-', lw=1, label='dive')
    ax.plot(climb[var], climb.depth, '-', lw=1, label='climb')
    ax.set_xlabel(label)
axs[0].set_ylim(300, 0)
axs[0].set_ylabel('Depth (m)')
axs[0].legend()
plt.show()

---
## ✏️ Your turn (Science group)

1. **Thermal lag:** Look at several dive–climb pairs. In which depth range do the dive and climb salinity disagree the most? Is that where temperature changes fastest? Interpolate the dive and climb onto the same depth grid (hint: `np.interp`, depth must be increasing) and plot the **difference** in salinity vs. depth. Average it over many pairs.
2. **Mixed layer depth:** For each deep profile in `binned`, find the depth where σ₀ first becomes 0.03 kg/m³ greater than at 10 m. Plot mixed layer depth against time. What happens between October and January? Why?
3. **Sound speed:** Plot `sound_speed` against depth for a deep profile. At what depth is sound slowest? Why does sound speed first decrease and then increase with depth? (Hint: temperature vs. pressure.)
4. **Eighteen Degree Water:** For each deep profile, find the thickness of the layer between 17.5 and 18.5 °C. Map it along the track. Where is it thickest?

---
## 7. Going further
* **More variables:** `list_glider_variables(SCIENCE_ID)` lists everything. Try `potential_temperature` or `conductivity`.
* **Currents:** the science dataset also has `u` and `v`, the depth-averaged current for each profile.
* **Engineering data:** Activity 2 looks at how the glider flew on the same dives.
* **Other gliders:** browse all datasets at <https://slocum-data.marine.rutgers.edu/erddap/tabledap/index.html>. Most Rutgers gliders have a `...-profile-sci-delayed` dataset just like this one.
* **Save your work:** `binned.to_csv('redwing_binned.csv')` writes a file you can download from the Colab file browser (folder icon on the left). Use *File → Save a copy in Drive* to keep your edits.